In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [3]:
df_reviews = spark.read.csv(
    'olist_order_reviews_dataset.csv', 
    header=True, 
    inferSchema=True, 
    multiLine=True,
    quote='"',
    escape='"'
)
df_reviews.show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                NULL|                  NULL| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                NULL|                  NULL| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                NULL|  Recebi bem antes ...| 2017-04-21 00:00:00|   

In [4]:
# # Remover pontuação da coluna review_comment_message
# from pyspark.sql.functions import col, regexp_replace


# df_reviews = df_reviews.withColumn(
#     "review_comment_message",
#     regexp_replace(col("review_comment_message"), "[^\w\s]", "")
# )
# df_reviews.show()

In [5]:
from pyspark.sql.functions import col

# Verificar se há order_ids nulos
null_order_id_count = df_reviews.filter(col("order_id").isNull()).count()

if null_order_id_count > 0:
    print(f"Foram encontrados {null_order_id_count} order_ids nulos.")
    df_reviews.filter(col("order_id").isNull()).show()
else:
    print("Não há order_ids nulos no DataFrame.")

Não há order_ids nulos no DataFrame.


In [6]:
from pyspark.sql.functions import col

# Contar o número de linhas antes da remoção
initial_count = df_reviews.count()

# Remover as linhas onde order_id é nulo
df_reviews = df_reviews.filter(col("order_id").isNotNull())

# Contar o número de linhas após a remoção
final_count = df_reviews.count()

removed_count = initial_count - final_count

if removed_count > 0:
    print(f"Foram removidas {removed_count} linhas com order_id nulo.")
    print(f"O DataFrame agora tem {final_count} linhas.")
else:
    print("Nenhuma linha com order_id nulo foi encontrada para remover.")


Nenhuma linha com order_id nulo foi encontrada para remover.


In [7]:
from pyspark.sql.functions import col, when, avg, concat_ws, collect_list

df_reviews = df_reviews.drop("review_id")
df_reviews = df_reviews.drop("review_comment_title")
df_reviews = df_reviews.drop("review_creation_date")
df_reviews = df_reviews.drop("review_answer_timestamp")


# Preencher valores nulos nos comentários com uma string vazia
df_reviews = df_reviews.withColumn(
    "review_comment_message",
    when(col("review_comment_message").isNull(), "").otherwise(col("review_comment_message"))
)
df_reviews.show()

# Agrupar por order_id, calcular a média do review_score e concatenar os comentários
df_reviews = df_reviews.groupBy("order_id").agg(
    avg("review_score").alias("review_score"),
    concat_ws(" | ", collect_list(when(col("review_comment_message") != "", col("review_comment_message")))).alias("review_comment_message")
)

df_reviews.show()


+--------------------+------------+----------------------+
|            order_id|review_score|review_comment_message|
+--------------------+------------+----------------------+
|73fc7af87114b3971...|           4|                      |
|a548910a1c6147796...|           5|                      |
|f9e4b658b201a9f2e...|           5|                      |
|658677c97b385a9be...|           5|  Recebi bem antes ...|
|8e6bfb81e283fa7e4...|           5|  Parabéns lojas la...|
|b18dcdf73be663668...|           1|                      |
|e48aa0d2dcec3a2e8...|           5|                      |
|c31a859e34e3adac2...|           5|                      |
|9c214ac970e842735...|           5|                      |
|b9bf720beb4ab3728...|           4|  aparelho eficient...|
|cdf9aa68e72324eeb...|           5|                      |
|3d374c9e46530bb5e...|           5|                      |
|9d6f15f95d01e79bd...|           4|  Mas um pouco ,tra...|
|2eaf8e099d871cd5c...|           4|                     

In [13]:
# Salvar o DataFrame agregado em CSV usando Pandas (evita dependencia do Hadoop no Windows)
import csv
import os
import pandas as pd
from datetime import datetime

reviews_final_pd = df_reviews.toPandas()
output_dir = os.getcwd()
output_path = os.path.join(output_dir, f"reviews_final_{datetime.now():%Y%m%d_%H%M%S}.csv")
reviews_final_pd.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\reviews_final_20260331_223131.csv
